# Task 3.1:  Explore Categorical Columns

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

df = pd.read_excel('Practice_Dataset.xlsx')

# List all categorical columns
cat_cols = df.select_dtypes(include='object').columns
print('Categorical columns:', list(cat_cols))

# Check unique values in each
for col in cat_cols:
    n_unique = df[col].nunique()
    print(f'\n{col} ({n_unique} unique):')
    print(df[col].value_counts())

Categorical columns: ['badge_number', 'name', 'position', 'work_date', 'in_time', 'out_time', 'status', 'department']

badge_number (30 unique):
badge_number
EMP1004    12
EMP1010    12
EMP1008    12
EMP1006    12
EMP1027    12
EMP1028    12
EMP1007    12
EMP1024    12
EMP1018    12
EMP1001    12
EMP1015    12
EMP1026    12
EMP1003    12
EMP1017    12
EMP1011    12
EMP1020    12
EMP1025    12
EMP1002    12
EMP1009    12
EMP1021    12
EMP1019    12
EMP1013    12
EMP1029    12
EMP1012    12
EMP1030    12
EMP1022    12
EMP1014    12
EMP1005    12
EMP1016    12
EMP1023    12
Name: count, dtype: int64

name (30 unique):
name
Omar Hassan              12
Saleh Al-Qahtani         12
Nasser Al-Dosari         12
Khalid Ibrahim           12
Lina Al-Nasser           12
Amal Al-Dosari           12
Yusuf Al-Bakr            12
Dalal Al-Rashidi         12
Abdulrahman Al-Subaie    12
Ahmed Al-Farsi           12
Majed Al-Zahrani         12
Sara Al-Hammad           12
Fatima Al-Sayed          12
Nawaf Al

# Task 3.2:  Label Encoding — For Ordinal Data

In [15]:
df = pd.read_excel('Practice_Dataset.xlsx')

# Label Encoding: converts each category to a number (0, 1, 2, ...)
# USE FOR: ordinal data where order matters, or binary columns
# DO NOT USE FOR: nominal data with 3+ categories -- creates fake order

le = LabelEncoder()
df_encoded = df.copy()

# status is binary (PRESENT/ABSENT) -> safe to Label Encode
df_encoded['status_encoded'] = le.fit_transform(df_encoded['status'])
print('Label mapping:')
for label, encoded in zip(le.classes_, range(len(le.classes_))):
    print(f'  {label} -> {encoded}')
print(df_encoded[['status', 'status_encoded']].head(10))

# Manual ordinal encoding for a real ordinal column (punch_level from Day 5)
ordinal_map = {'Low': 0, 'Medium': 1, 'High': 2}
punch_level = pd.cut(df['punch_count'], bins=[-1, 2, 4, float('inf')],
                      labels=['Low', 'Medium', 'High'])
df_encoded['punch_level_encoded'] = punch_level.map(ordinal_map)
print(df_encoded[['punch_count', 'punch_level_encoded']].head())

Label mapping:
  ABSENT -> 0
  PRESENT -> 1
    status  status_encoded
0  PRESENT               1
1  PRESENT               1
2  PRESENT               1
3  PRESENT               1
4  PRESENT               1
5  PRESENT               1
6  PRESENT               1
7  PRESENT               1
8  PRESENT               1
9  PRESENT               1
   punch_count punch_level_encoded
0            4                   1
1            4                   1
2            4                   1
3            6                   2
4            2                   0


# Task 3.3: One-Hot Encoding 

In [16]:
df = pd.read_excel('Practice_Dataset.xlsx')

# One-Hot: creates a new binary column for each category
# USE FOR: nominal data where no order exists (position, department)
# WATCH OUT: too many categories = too many columns

# Method 1: pandas get_dummies (easiest)
df_onehot = pd.get_dummies(df, columns=['position'], prefix='pos')
print('Columns after one-hot encoding:')
print(df_onehot.columns.tolist())

# Method 2: drop_first=True to avoid multicollinearity
df_onehot_drop = pd.get_dummies(df, columns=['position'], prefix='pos', drop_first=True)
print('\nWith drop_first=True:')
print(df_onehot_drop.columns.tolist())

Columns after one-hot encoding:
['badge_number', 'name', 'work_date', 'in_time', 'out_time', 'punch_count', 'status', 'hours_worked', 'department', 'monthly_salary', 'satisfaction_score', 'is_absent', 'pos_ACCOUNTANT', 'pos_ADMINISTRATOR', 'pos_CLERK', 'pos_DRIVER', 'pos_ENGINEER', 'pos_LABORER', 'pos_SUPERVISOR', 'pos_TECHNICIAN']

With drop_first=True:
['badge_number', 'name', 'work_date', 'in_time', 'out_time', 'punch_count', 'status', 'hours_worked', 'department', 'monthly_salary', 'satisfaction_score', 'is_absent', 'pos_ADMINISTRATOR', 'pos_CLERK', 'pos_DRIVER', 'pos_ENGINEER', 'pos_LABORER', 'pos_SUPERVISOR', 'pos_TECHNICIAN']


# Task 3.4: Frequency & Target Encoding

In [17]:
df = pd.read_excel('Practice_Dataset.xlsx')

# Frequency Encoding: replace category with its count or frequency
# Good for high-cardinality columns (many unique values)
freq_map = df['position'].value_counts(normalize=True)
df_freq = df.copy()
df_freq['position_freq'] = df_freq['position'].map(freq_map)
print('Frequency encoding:')
print(df_freq[['position', 'position_freq']].drop_duplicates())

# Target Encoding: replace category with the mean of the target variable
# CAUTION: can cause data leakage if not done carefully
target_means = df.groupby('position')['monthly_salary'].mean()
df_freq['position_target'] = df_freq['position'].map(target_means)
print('\nTarget encoding (mean monthly_salary per position):')
print(df_freq[['position', 'position_target']].drop_duplicates())

Frequency encoding:
         position  position_freq
0      TECHNICIAN       0.133333
1   ADMINISTRATOR       0.133333
2      ACCOUNTANT       0.100000
3      SUPERVISOR       0.133333
4        ENGINEER       0.133333
6           CLERK       0.100000
10        LABORER       0.133333
28         DRIVER       0.133333

Target encoding (mean monthly_salary per position):
         position  position_target
0      TECHNICIAN      6598.355556
1   ADMINISTRATOR      6942.260870
2      ACCOUNTANT      8624.441176
3      SUPERVISOR     11455.543478
4        ENGINEER     11772.446809
6           CLERK      7944.333333
10        LABORER      7948.891304
28         DRIVER      8737.893617


# Task 3.5: Encoding Decision Guide

In [18]:
# ENCODING CHEAT SHEET (applied to this dataset):
#
# status (binary, 2 values)          -> Label Encoding (0/1)          [Task 3.2]
# punch_level (ordinal, has order)   -> Ordinal Encoding               [Task 3.2]
# position, department (nominal,     -> One-Hot Encoding (get_dummies) [Task 3.3]
#   few categories: 8 and 4)
# badge_number, name (nominal,       -> Frequency/Target Encoding, or  [Task 3.4]
#   high cardinality: 30 unique)        drop entirely -- pure identifiers
# in_time, out_time (152/191 unique) -> Not for encoding -- convert to
#                                        datetime and extract features instead
#
# ALWAYS document which encoding you used and WHY